In [47]:
import numpy as np
import math as m

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

from scipy.optimize import minimize

import matplotlib.pyplot as plt

# exact expectation values come straight from Statevector amplitudes (no simulator
# needed); qasm_sim is only for the "measured" (shot-sampled) path.
qasm_sim = AerSimulator()

In [48]:
def cut_count(z, edges):
    return sum((1-z[i]*z[j])/2 for i, j in edges)

def all_costs(n, edges):
    costs = []
    for state in range(2**n): # for n nodes there are 2^n possible arrangements as each node can take -1 or +1 colour
        state_as_bits = format(state, f'0{n}b')[::-1] # convert state to binary
        z = [1 if b=='0' else -1 for b in state_as_bits] # convert binary to +1/-1
        costs.append(cut_count(z, edges))

    return costs

verts = [0,1,2,3,4,5]
edges = [[0,1], [0,2], [0,5], [1,2], [1,3], [2, 3], [2,4], [3,5]]
costs = all_costs(len(verts), edges)
best_cost = max(costs)
best_cost_index = costs.index(best_cost)
best_cost_index_binary = format(best_cost_index, f'0{len(verts)}b')
print("All costs:", costs)
print("Max cut:", best_cost)
print(f"Best cost index: {best_cost_index}, binary: {best_cost_index_binary}")



All costs: [0.0, 3.0, 3.0, 4.0, 4.0, 5.0, 5.0, 4.0, 3.0, 6.0, 4.0, 5.0, 5.0, 6.0, 4.0, 3.0, 1.0, 4.0, 4.0, 5.0, 3.0, 4.0, 4.0, 3.0, 4.0, 7.0, 5.0, 6.0, 4.0, 5.0, 3.0, 2.0, 2.0, 3.0, 5.0, 4.0, 6.0, 5.0, 7.0, 4.0, 3.0, 4.0, 4.0, 3.0, 5.0, 4.0, 4.0, 1.0, 3.0, 4.0, 6.0, 5.0, 5.0, 4.0, 6.0, 3.0, 4.0, 5.0, 5.0, 4.0, 4.0, 3.0, 3.0, 0.0]
Max cut: 7.0
Best cost index: 25, binary: 011001


In [ ]:
# quantum impl of the methods for QAOA

def problem_hamiltonian_operator(qc, edges, gamma): # equivalent of cut_count operator above the sum in the above operator becomes a product of exponentials check the c(x) to operator proof. THIS IS THE PROBLEM HAMILTONIAN implemented as a unitary operator
    for i, j in edges:
        qc.cx(i, j)
        qc.rz(-gamma, j)
        qc.cx(i,j)

def mixer_hamiltonian_operator(qc, beta):
    for i in range(qc.num_qubits):
        qc.rx(beta, i)

def append_qaoa_hamiltonians(qc, edges, gamma, beta):
    # problem Hamiltonian
    problem_hamiltonian_operator(qc, edges, gamma)
    # mixer hamiltonian
    mixer_hamiltonian_operator(qc, beta)

def create_qaoa_circuit(n, edges, gammas, betas):
    qc = QuantumCircuit(n)
    # initial state is uniform superposition
    qc.h(range(n))

    for gamma, beta in zip(gammas, betas):
        append_qaoa_hamiltonians(qc, edges, gamma, beta)

    return qc

def expectation_exact(qc, costs):
    # statevector in this simulaition will hold all of the probablities of all of the 2^n coloring comninations then u multiply with costs we already clacluated so when do its weighted sum for each of the coliring options so that gives expecation value. now if we drive the expecation value then one of the coloring option has to have more probebality than other so that average can go up. nelder mead optim will try to move the average up which means the max value in the 2^n options is getting higher weightage. which is the way we find the max cut value
    state_vector = Statevector.from_instruction(qc)
    max_prob_index = np.argmax(state_vector.probabilities())
    probs_all_colorings = state_vector.probabilities_dict()
    expectation = 0
    for coloring, prob in probs_all_colorings.items():
        expectation += prob * costs[int(coloring, 2)]
    return expectation, max_prob_index

def get_expectation_for_optimization(params, edges, costs, n):
    param_len = len(params) // 2
    gammas = params[:param_len]
    betas = params[param_len:]
    qc = create_qaoa_circuit(n, edges, gammas, betas)
    expectation, _ = expectation_exact(qc, costs)
    return -expectation # we want to maximize expectation, but scipy minimize minimizes, so we return the negative of the expectation value


In [50]:
gamma = 1.0
beta = 1.0
qc = create_qaoa_circuit(len(verts), edges, [gamma], [beta])
expectation, max_prob_index = expectation_exact(qc, costs)

print(f"Expectation value: {expectation:.4f} \n and max index in quantum is {max_prob_index} with binary representation {format(max_prob_index, f'0{len(verts)}b')}")

result = minimize(get_expectation_for_optimization, x0=[1.0, 1.0], args=(edges, costs, len(verts)), method='Nelder-Mead')

print(f"Optimized gamma: {result.x[0]:.4f}")
print(f"Optimized beta: {result.x[1]:.4f}")
print(f"Optimized expectation: {-result.fun:.4f}")


Expectation value: 4.7167 
 and max index in quantum is 25 with binary representation 011001
Optimized gamma: 0.6016
Optimized beta: 0.6939
Optimized expectation: 5.2956


In [51]:
# as noticed above expecation got to only 5.2 or so we ned 7 which means the state which represents the best formation needs to have highest amplitude and rest near 0 which is hard with only a cricuit with 1 layer of problem hamiltonian and 1 layer of mixer hamiltonian. so we need to increase the number of layers to get better results. so we will try with 2 layers and see if we can get better results.

for depth in [1, 2, 3, 4, 5]:
    x0 = [1.0] * (depth*2)
    result = minimize(get_expectation_for_optimization, x0=x0, args=(edges, costs, len(verts)), method='Nelder-Mead')
    print(f"Depth: {depth}")
    print(f"best expectation value: {-result.fun:.4f}")


Depth: 1
best expectation value: 5.2956
Depth: 2
best expectation value: 6.0589
Depth: 3
best expectation value: 6.4362
Depth: 4
best expectation value: 6.6885
Depth: 5
best expectation value: 6.7214
